# PDF Study Assistant (Colab)
Runtime > Change runtime type > **T4 GPU**, then run the cells in order (1 -> 5).
Cell 6 = interactive quiz (choose number of questions, then press Enter to start).
Generates MCQs, 2-mark Q&A, long answers, fill-in-the-blanks and topic-wise summaries, grounded only in your PDF/text.

In [ ]:
# CELL 1: install (run once, ~2 min)
!pip -q install pymupdf transformers accelerate bitsandbytes reportlab pytesseract pillow
!apt-get -qq install -y tesseract-ocr > /dev/null

In [ ]:
# CELL 2: load the AI model (first run downloads ~5 GB)
import torch, json, re, math
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"   # for a faster/lighter option use "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")

SYS = ("You are an exam-preparation assistant. Use ONLY the study text provided. "
       "Never add facts that are not in the text. If the text does not support something, leave it out.")

def ask(prompt, max_new_tokens=1800):
    msgs = [{"role": "system", "content": SYS}, {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.3, top_p=0.9)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("Model ready")

In [ ]:
# CELL 3: give your material  (PDF upload OR pasted text)
#@title Input
SOURCE = "pdf"  #@param ["pdf","text"]
PASTED_TEXT = ""  #@param {type:"string"}

import fitz, pytesseract
from PIL import Image
from collections import Counter

def read_pdf(path):
    doc = fitz.open(path)
    lines = []   # (page, font_size, text)
    for p in doc:
        got = False
        for b in p.get_text("dict")["blocks"]:
            for l in b.get("lines", []):
                t = "".join(s["text"] for s in l["spans"]).strip()
                if t:
                    lines.append((p.number + 1, max(s["size"] for s in l["spans"]), t)); got = True
        if not got:   # scanned page -> OCR
            pix = p.get_pixmap(dpi=200)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            for t in pytesseract.image_to_string(img).split("\n"):
                if t.strip(): lines.append((p.number + 1, 0, t.strip()))
    print("Pages:", len(doc)); return lines

def build_topics(lines):
    sizes = [round(s) for _, s, _ in lines if s > 0]
    body = Counter(sizes).most_common(1)[0][0] if sizes else 0
    topics, cur = [], None
    for pg, s, t in lines:
        if body and s >= body + 2 and len(t) < 100:          # heading
            if cur and not cur["text"].strip(): cur["title"] += " " + t
            else:
                cur = {"title": t, "pages": [pg], "text": ""}; topics.append(cur)
        else:
            if cur is None:
                cur = {"title": "Introduction", "pages": [pg], "text": ""}; topics.append(cur)
            cur["text"] += t + "\n"
            if pg not in cur["pages"]: cur["pages"].append(pg)
    merged = []
    for tp in topics:                                         # merge tiny topics
        if merged and len(tp["text"].split()) < 60:
            merged[-1]["text"] += tp["title"] + "\n" + tp["text"]; merged[-1]["pages"] += tp["pages"]
        else: merged.append(tp)
    if len(merged) < 2:                                       # fallback: 3-page chunks
        by_page = {}
        for pg, _, t in lines: by_page.setdefault(pg, []).append(t)
        pgs = sorted(by_page); merged = []
        for i in range(0, len(pgs), 3):
            grp = pgs[i:i+3]
            merged.append({"title": f"Pages {grp[0]}-{grp[-1]}", "pages": grp,
                           "text": "\n".join("\n".join(by_page[p]) for p in grp)})
    for tp in merged:
        tp["pages"] = f"{min(tp['pages'])}-{max(tp['pages'])}"
    return merged

if SOURCE == "pdf":
    from google.colab import files
    up = files.upload()
    all_lines = []
    for name in up: all_lines += read_pdf(name)
    TOPICS = build_topics(all_lines)
else:
    assert PASTED_TEXT.strip(), "Paste your text in PASTED_TEXT above"
    TOPICS = build_topics([(1, 0, l) for l in PASTED_TEXT.split("\n") if l.strip()])

print("\nTABLE OF CONTENTS")
for i, t in enumerate(TOPICS, 1):
    print(f"{i:>3}. {t['title']}  (pages {t['pages']}, {len(t['text'].split())} words)")

In [ ]:
# CELL 4: choose what to generate, then run
#@title Options
TOPIC_NUMBERS = "all"  #@param {type:"string"}
WHAT = "MCQ"  #@param ["MCQ","2-mark Q&A","Long answers","Fill in the blanks","Summary"]
COUNT = 10  #@param {type:"integer"}
DIFFICULTY = "medium"  #@param ["easy","medium","hard"]
MARKS = 5  #@param [5,10]
SUMMARY_LEVEL = "medium"  #@param ["short","medium","detailed"]

sel = TOPICS if TOPIC_NUMBERS.strip().lower() == "all" else \
      [TOPICS[int(i) - 1] for i in TOPIC_NUMBERS.split(",")]

def chunks_of(topics, size=1400):
    out = []
    for t in topics:
        w = t["text"].split()
        for i in range(0, len(w), size):
            piece = " ".join(w[i:i+size])
            if len(piece.split()) > 40:
                out.append({"title": t["title"], "pages": t["pages"], "text": piece})
    return out

def parse_json(s):
    try: return json.loads(s[s.index("["): s.rindex("]") + 1])
    except Exception: return []

P = {
"MCQ": """Create __N__ multiple-choice questions (difficulty: __D__) from the text below.
Return ONLY a JSON list: [{"question":"...","options":["...","...","...","..."],"answer":"A","explanation":"one line from the text"}]
Exactly 4 options, answer is a letter A-D. No duplicate or ambiguous questions.
TEXT:
__T__""",
"2-mark Q&A": """Create __N__ two-mark exam questions (difficulty: __D__) with concise 2-3 sentence answers from the text.
Cover definitions, differences, key facts. Return ONLY a JSON list: [{"question":"...","answer":"..."}]
TEXT:
__T__""",
"Long answers": """Create __N__ __M__-mark long-answer questions (difficulty: __D__) from the text.
Each answer must be structured (intro, numbered points with explanation, short conclusion), about __W__ words, using only the text.
Return ONLY a JSON list: [{"question":"...","answer":"..."}]
TEXT:
__T__""",
"Fill in the blanks": """Create __N__ fill-in-the-blank questions (difficulty: __D__) from the text, blanking a key term/definition/fact.
Write the blank as _____ . Return ONLY a JSON list: [{"question":"sentence with _____","answer":"missing word"}]
TEXT:
__T__""",
}

def gen_items(kind, chunks, n, diff=None):
    step = max(1, len(chunks) // max(1, n)) if len(chunks) > n else 1
    chunks = chunks[::step][:max(n, 1)] if len(chunks) > n else chunks
    per = math.ceil(n / len(chunks)); items = []
    for c in chunks:
        pr = (P[kind].replace("__N__", str(per)).replace("__D__", diff or DIFFICULTY).replace("__M__", str(MARKS))
              .replace("__W__", "150-200" if MARKS == 5 else "350-400").replace("__T__", c["text"]))
        res = parse_json(ask(pr, 2500 if kind == "Long answers" else 1800)) or parse_json(ask(pr))
        for r in res:
            if isinstance(r, dict) and r.get("question"):
                r["ref"] = (f"p.{c['pages']}" if c["title"].startswith("Pages ") else f"{c['title']}, p.{c['pages']}"); items.append(r)
    seen, uniq = set(), []
    for r in items:
        k = r["question"].lower()[:60]
        if k not in seen: seen.add(k); uniq.append(r)
    return uniq[:n]

def summarize(topic, level):
    words = {"short": "60-100", "medium": "150-250", "detailed": "350-500"}[level]
    parts = [ask(f"Summarize this study text in {words} words as bullet points. Then list 'Key terms' with one-line definitions.\n\nTEXT:\n{c['text']}", 1200)
             for c in chunks_of([topic])]
    if len(parts) > 1:
        return ask("Merge these partial summaries into ONE clean topic summary (" + words + " words) with 'Key terms' at the end:\n\n" + "\n\n".join(parts), 1500)
    return parts[0] if parts else ""

paper, key = "", ""
if WHAT == "Summary":
    paper = "TOPIC-WISE SUMMARY\n\n"
    for t in sel:
        print("Summarizing:", t["title"])
        paper += f"## {t['title']} (pages {t['pages']})\n{summarize(t, SUMMARY_LEVEL)}\n\n"
    if len(sel) > 1:
        paper += "## Combined overview\n" + ask("Write a 150-word overview of the whole material from these summaries:\n\n" + paper, 800)
else:
    items = gen_items(WHAT, chunks_of(sel), COUNT)
    if WHAT == "MCQ":
        items = [q for q in items if isinstance(q.get("options"), list) and len(q["options"]) == 4
                 and str(q.get("answer", "")).strip().upper()[:1] in "ABCD"]
        paper = "MCQ QUESTION PAPER\n\n"; key = "MCQ ANSWER KEY\n\n"
        for i, q in enumerate(items, 1):
            paper += f"Q{i}. {q['question']}\n" + "".join(f"   {'ABCD'[j]}) {o}\n" for j, o in enumerate(q["options"])) + "\n"
            key += f"{i}. {q['answer'].strip().upper()[:1]} - {q.get('explanation','')} [{q['ref']}]\n"
    elif WHAT == "Fill in the blanks":
        paper = "FILL IN THE BLANKS\n\n"; key = "ANSWER KEY\n\n"
        for i, q in enumerate(items, 1):
            paper += f"{i}. {q['question']}\n\n"; key += f"{i}. {q['answer']} [{q['ref']}]\n"
    else:
        m = 2 if WHAT == "2-mark Q&A" else MARKS
        paper = f"{WHAT.upper()} ({m} marks each)\n\n"
        for i, q in enumerate(items, 1):
            paper += f"Q{i}. {q['question']}  [{m} marks]\nAns: {q.get('answer','')}\n(Source: {q['ref']})\n\n"

print(paper); print(key)

In [ ]:
# CELL 5: download as PDF
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from xml.sax.saxutils import escape
from google.colab import files

def make_pdf(text, fname, title):
    st = getSampleStyleSheet()
    story = []
    for line in text.split("\n"):
        if line.startswith("## "): story.append(Paragraph(escape(line[3:]), st["Heading2"]))
        elif not line.strip(): story.append(Spacer(1, 6))
        else: story.append(Paragraph(escape(line).replace("   ", "&nbsp;&nbsp;&nbsp;"), st["BodyText"]))
    def footer(c, d):
        c.setFont("Helvetica", 8); c.drawString(40, 25, title); c.drawRightString(555, 25, f"Page {d.page}")
    SimpleDocTemplate(fname, pagesize=A4).build(story, onFirstPage=footer, onLaterPages=footer)

tag = WHAT.replace(" ", "_").replace("&", "and")
make_pdf(paper, f"{tag}.pdf", WHAT); files.download(f"{tag}.pdf")
if key:
    make_pdf(key, f"{tag}_answer_key.pdf", WHAT + " - Answer Key"); files.download(f"{tag}_answer_key.pdf")

In [ ]:
# CELL 6: interactive quiz (run Cell 3 first; Cells 4-5 are not needed)
#@title Quiz settings
QUIZ_TOPICS = "all"  #@param {type:"string"}
QUIZ_COUNT = 10  #@param {type:"integer"}
QUIZ_DIFFICULTY = "medium"  #@param ["easy","medium","hard"]

qsel = TOPICS if QUIZ_TOPICS.strip().lower() == "all" else [TOPICS[int(i) - 1] for i in QUIZ_TOPICS.split(",")]

def valid_mcq(q):
    return (isinstance(q.get("options"), list) and len(q["options"]) == 4
            and str(q.get("answer", "")).strip().upper()[:1] in list("ABCD"))

print(f"Preparing {QUIZ_COUNT} questions ({QUIZ_DIFFICULTY})...")
quiz, seen = [], set()
for attempt in range(3):                      # top-up until we reach the requested count
    need = QUIZ_COUNT - len(quiz)
    if need <= 0: break
    for q in gen_items("MCQ", chunks_of(qsel), need + 2, QUIZ_DIFFICULTY):
        k = q["question"].lower()[:60]
        if valid_mcq(q) and k not in seen and len(quiz) < QUIZ_COUNT:
            seen.add(k); quiz.append(q)

if not quiz:
    print("Could not generate questions. Run this cell again.")
else:
    print(f"\nQuiz ready: {len(quiz)} questions. Answer A/B/C/D, or Q to quit.")
    go = input("Press Enter to START (or type Q to cancel): ").strip().upper()
    if go != "Q":
        score, wrong, quit_ = 0, [], False
        for i, q in enumerate(quiz, 1):
            print(f"\nQuestion {i}/{len(quiz)}   (score so far: {score})")
            print(q["question"])
            for j, o in enumerate(q["options"]): print(f"   {'ABCD'[j]}) {o}")
            while True:
                a = input("Your answer (A/B/C/D, Q=quit): ").strip().upper()[:1]
                if a in list("ABCD") or a == "Q": break
                print("Please type A, B, C or D.")
            if a == "Q": quit_ = True; break
            right = q["answer"].strip().upper()[:1]
            if a == right: score += 1; print("Correct!", q.get("explanation", ""))
            else: wrong.append((i, q, a)); print(f"Wrong. Correct answer: {right}.", q.get("explanation", ""))
        done = i if quit_ else len(quiz)
        print("\n" + "=" * 40)
        print(f"FINAL SCORE: {score}/{done}  ({100*score//max(done,1)}%)")
        if wrong:
            print("\nREVIEW YOUR MISTAKES")
            for i, q, a in wrong:
                r = q["answer"].strip().upper()[:1]
                print(f"Q{i}. {q['question']}\n   You: {a}) {q['options']['ABCD'.index(a)]}\n   Correct: {r}) {q['options']['ABCD'.index(r)]}\n   [{q['ref']}]\n")
        print("Run this cell again for a new quiz.")